# Teste das funções de `data.py`

Este notebook serve para validar manualmente as funções de carga, preparação e separação dos dados antes da criação dos testes unitários.

In [1]:
import importlib

import data_ingestion
import experiment_tracker
import model_evaluation
import model_trainer

importlib.reload(data_ingestion)
importlib.reload(model_trainer)
importlib.reload(model_evaluation)
importlib.reload(experiment_tracker)

from data_ingestion import DataIngestion
from experiment_tracker import log_experiment, log_metrics_experiment
from model_evaluation import evaluate_model
from model_trainer import (
    cross_validate_model,
    fit_model,
    predict,
    predict_proba,
)

In [2]:
df = DataIngestion().load_data()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   customer_id        7043 non-null   object
 1   count              7043 non-null   int64 
 2   country            7043 non-null   object
 3   state              7043 non-null   object
 4   city               7043 non-null   object
 5   zip_code           7043 non-null   int64 
 6   lat_long           7043 non-null   object
 7   latitude           7043 non-null   object
 8   longitude          7043 non-null   object
 9   gender             7043 non-null   object
 10  senior_citizen     7043 non-null   object
 11  partner            7043 non-null   object
 12  dependents         7043 non-null   object
 13  tenure_months      7043 non-null   int64 
 14  phone_service      7043 non-null   object
 15  multiple_lines     7043 non-null   object
 16  internet_service   7043 non-null   object


In [3]:
df = DataIngestion().treat_data(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 50 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   customer_id                             7043 non-null   object 
 1   count                                   7043 non-null   int64  
 2   country                                 7043 non-null   object 
 3   state                                   7043 non-null   object 
 4   city                                    7043 non-null   object 
 5   zip_code                                7043 non-null   int64  
 6   lat_long                                7043 non-null   object 
 7   latitude                                7043 non-null   object 
 8   longitude                               7043 non-null   object 
 9   gender                                  7043 non-null   object 
 10  senior_citizen                          7043 non-null   int3

In [4]:
ingestion = DataIngestion()

df = ingestion.load_data()
df = ingestion.treat_data(df)
X_train, X_test, y_train, y_test = ingestion.split_train_test(df)
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5634 entries, 4626 to 6017
Data columns (total 27 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   senior_citizen                          5634 non-null   int32  
 1   partner                                 5634 non-null   int32  
 2   dependents                              5634 non-null   int32  
 3   tenure_months                           5634 non-null   int32  
 4   is_new_customer                         5634 non-null   int32  
 5   phone_services                          5634 non-null   int32  
 6   multiples_lines                         5634 non-null   int32  
 7   internet_dsl                            5634 non-null   int32  
 8   internet_fiber                          5634 non-null   int32  
 9   online_security                         5634 non-null   int32  
 10  online_backup                           5634 non-null   int32 

In [5]:
ingestion = DataIngestion()

df = ingestion.load_data()
df = ingestion.treat_data(df)

X_train, X_test, y_train, y_test = ingestion.split_train_test(df)

In [6]:
# Validacao cruzada estratificada da MLP.
# Use apenas X_train/y_train aqui;
# X_test/y_test fica reservado para a avaliacao final.
cv_results, cv_summary = cross_validate_model(
    X=X_train,
    y=y_train,
    n_splits=5,
    hidden_layers=(64, 32),
    dropout=0.1,
    learning_rate=0.001,
    epochs=50,
    batch_size=64,
)

cv_results

,fold,train_size,valid_size,final_train_loss,pr_auc
0,1,4507,1127,0.330302,0.632170
1,2,4507,1127,0.335403,0.634628
2,3,4507,1127,0.329197,0.653735
3,4,4507,1127,0.330475,0.655348
4,5,4508,1126,0.335160,0.659247


In [7]:
cv_summary

,final_train_loss,pr_auc
mean,0.332108,0.647026
std,0.002940,0.012629


In [8]:
model, scaler, losses = fit_model(
    X_train=X_train,
    y_train=y_train,
    hidden_layers=(64, 32),
    epochs=50,
    learning_rate=0.001,
)

metrics = evaluate_model(
    model=model,
    scaler=scaler,
    X_test=X_test,
    y_test=y_test,
)

metrics

{'pr_auc': 0.6410953161428167}

In [9]:
from data_ingestion import DataIngestion
from experiment_tracker import log_experiment
from model_evaluation import evaluate_model
from model_trainer import fit_model

ingestion = DataIngestion()

df = ingestion.load_data()
df = ingestion.treat_data(df)
X_train, X_test, y_train, y_test = ingestion.split_train_test(df)

model, scaler, losses = fit_model(X_train, y_train)

metrics = evaluate_model(model, scaler, X_test, y_test)

log_experiment(
    model=model,
    scaler=scaler,
    metrics=metrics,
    losses=losses,
    params={
        "hidden_layers": "(64, 32)",
        "learning_rate": 0.01,
        "epochs": 50,
    },
    experiment_name="churn_experimentacao_pr_auc",
)

2026/06/21 16:48:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/21 16:48:18 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/21 16:48:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/21 16:48:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.h

'cdd52f989fcf4865a8ced7da03b572be'

In [10]:
import pandas as pd

experiments = [
    {
        "run_name": "mlp_64_32_lr_001",
        "hidden_layers": (64, 32),
        "learning_rate": 0.001,
        "epochs": 50,
        "dropout": 0.0,
    },
    {
        "run_name": "mlp_128_64_lr_001",
        "hidden_layers": (128, 64),
        "learning_rate": 0.001,
        "epochs": 50,
        "dropout": 0.0,
    },
    {
        "run_name": "mlp_128_64_dropout",
        "hidden_layers": (128, 64),
        "learning_rate": 0.005,
        "epochs": 50,
        "dropout": 0.2,
    },
    {
        "run_name": "mlp_64_32_lr_01",
        "hidden_layers": (64, 32),
        "learning_rate": 0.01,
        "epochs": 50,
        "dropout": 0.0,
    },
]

In [11]:
results = []

for config in experiments:
    model, scaler, losses = fit_model(
        X_train=X_train,
        y_train=y_train,
        hidden_layers=config["hidden_layers"],
        learning_rate=config["learning_rate"],
        epochs=config["epochs"],
        dropout=config["dropout"],
    )

    metrics = evaluate_model(
        model=model,
        scaler=scaler,
        X_test=X_test,
        y_test=y_test,
    )

    run_id = log_experiment(
        model=model,
        scaler=scaler,
        metrics=metrics,
        losses=losses,
        params={
            "hidden_layers": str(config["hidden_layers"]),
            "learning_rate": config["learning_rate"],
            "epochs": config["epochs"],
            "dropout": config["dropout"],
        },
        experiment_name="churn_experimentacao_pr_auc",
        run_name=config["run_name"],
    )

    results.append(
        {
            "run_id": run_id,
            "run_name": config["run_name"],
            **config,
            **metrics,
        }
    )

df_results = pd.DataFrame(results)
df_results.sort_values("pr_auc", ascending=False)

2026/06/21 16:48:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/21 16:48:31 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/21 16:48:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/21 16:48:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.h

,run_id,run_name,hidden_layers,learning_rate,epochs,dropout,pr_auc
0,269f5dec805845aca334e490ae9f654b,mlp_64_32_lr_001,"(64, 32)",0.001,50,0.0,0.641095
2,578f16a3a39743819fb7cfd418a55cd7,mlp_128_64_dropout,"(128, 64)",0.005,50,0.2,0.623782
1,0012092d89604600bcca8b725e5675b8,mlp_128_64_lr_001,"(128, 64)",0.001,50,0.0,0.603983
3,f92e2387fe4842509e1e16393be9a156,mlp_64_32_lr_01,"(64, 32)",0.010,50,0.0,0.565389


In [ ]:
import pandas as pd

experiments = [
    {
        "run_name": "mlp_128_64_32_lr_001",
        "hidden_layers": (128, 64, 32),
        "learning_rate": 0.001,
        "epochs": 50,
        "dropout": 0.0,
    },
    {
        "run_name": "mlp_128_64_32_lr_01",
        "hidden_layers": (128, 64, 32),
        "learning_rate": 0.01,
        "epochs": 50,
        "dropout": 0.0,
    },
    {
        "run_name": "mlp_256_128_dropout",
        "hidden_layers": (256, 128),
        "learning_rate": 0.001,
        "epochs": 50,
        "dropout": 0.2,
    },
    {
        "run_name": "mlp_1056_512_256_lr_01",
        "hidden_layers": (1056, 512, 256),
        "learning_rate": 0.001,
        "epochs": 50,
        "dropout": 0.1,
    },
]

In [8]:
results = []

for config in experiments:
    model, scaler, losses = fit_model(
        X_train=X_train,
        y_train=y_train,
        hidden_layers=config["hidden_layers"],
        learning_rate=config["learning_rate"],
        epochs=config["epochs"],
        dropout=config["dropout"],
    )

    metrics = evaluate_model(
        model=model,
        scaler=scaler,
        X_test=X_test,
        y_test=y_test,
    )

    run_id = log_experiment(
        model=model,
        scaler=scaler,
        metrics=metrics,
        losses=losses,
        params={
            "hidden_layers": str(config["hidden_layers"]),
            "learning_rate": config["learning_rate"],
            "epochs": config["epochs"],
            "dropout": config["dropout"],
        },
        experiment_name="churn_experimentacao_pr_auc",
        run_name=config["run_name"],
    )

    results.append(
        {
            "run_id": run_id,
            "run_name": config["run_name"],
            **config,
            **metrics,
        }
    )

df_results = pd.DataFrame(results)
df_results.sort_values("pr_auc", ascending=False)

2026/06/10 20:09:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 20:09:29 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/10 20:09:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 20:09:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.h

,run_id,run_name,hidden_layers,learning_rate,epochs,dropout,accuracy,precision,recall,f1_score,roc_auc
2,bee3e10250134b9e9240a0b5f2c6114b,mlp_256_128_dropout,"(256, 128)",0.001,50,0.2,0.777857,0.588921,0.540107,0.563459,0.833672
0,ddf6f1ca227d49a6b141eb6844c59c58,mlp_128_64_32_lr_001,"(128, 64, 32)",0.001,50,0.0,0.775727,0.580556,0.558824,0.569482,0.809055
1,547463fa4f3d40a19b8704524eb7ae31,mlp_128_64_32_lr_01,"(128, 64, 32)",0.010,50,0.0,0.754436,0.533654,0.593583,0.562025,0.802749
3,cf527d80fb8c459b8fbc4898db9ede4e,mlp_1056_512_256_lr_01,"(1056, 512, 256)",0.001,50,0.1,0.760823,0.552113,0.524064,0.537723,0.793800


In [9]:
import pandas as pd
from experiment_tracker import log_experiment

# Novos experimentos focados nos melhores resultados observados no MLflow.
# A arquitetura (64, 32) com learning_rate=0.001 teve o melhor PR-AUC ate agora.
# Aqui testamos variacoes proximas e registramos
# metricas de treino/teste para observar overfitting.
experiments = [
    {
        "run_name": "mlp_64_32_lr_0005_epochs_100",
        "hidden_layers": (64, 32),
        "learning_rate": 0.0005,
        "epochs": 100,
        "dropout": 0.0,
    },
    {
        "run_name": "mlp_64_32_lr_001_dropout_01_epochs_100",
        "hidden_layers": (64, 32),
        "learning_rate": 0.001,
        "epochs": 100,
        "dropout": 0.1,
    },
    {
        "run_name": "mlp_32_16_lr_001_epochs_75",
        "hidden_layers": (32, 16),
        "learning_rate": 0.001,
        "epochs": 75,
        "dropout": 0.0,
    },
    {
        "run_name": "mlp_96_48_lr_001_dropout_01_epochs_75",
        "hidden_layers": (96, 48),
        "learning_rate": 0.001,
        "epochs": 75,
        "dropout": 0.1,
    },
]

results = []

for config in experiments:
    model, scaler, losses = fit_model(
        X_train=X_train,
        y_train=y_train,
        hidden_layers=config["hidden_layers"],
        learning_rate=config["learning_rate"],
        epochs=config["epochs"],
        dropout=config["dropout"],
    )

    train_metrics = evaluate_model(
        model=model,
        scaler=scaler,
        X_test=X_train,
        y_test=y_train,
    )
    test_metrics = evaluate_model(
        model=model,
        scaler=scaler,
        X_test=X_test,
        y_test=y_test,
    )

    mlflow_metrics = {
        **{f"train_{key}": value for key, value in train_metrics.items()},
        **{f"test_{key}": value for key, value in test_metrics.items()},
        # Metricas sem prefixo para ordenar no MLflow.
        **test_metrics,
        "pr_auc_gap": train_metrics["pr_auc"] - test_metrics["pr_auc"],
    }

    run_id = log_experiment(
        model=model,
        scaler=scaler,
        metrics=mlflow_metrics,
        losses=losses,
        params={
            "hidden_layers": str(config["hidden_layers"]),
            "learning_rate": config["learning_rate"],
            "epochs": config["epochs"],
            "dropout": config["dropout"],
        },
        experiment_name="churn_experimentacao_pr_auc",
        run_name=config["run_name"],
    )

    results.append(
        {
            "run_id": run_id,
            "run_name": config["run_name"],
            **config,
            "train_pr_auc": train_metrics["pr_auc"],
            "test_pr_auc": test_metrics["pr_auc"],
            "pr_auc_gap": train_metrics["pr_auc"] - test_metrics["pr_auc"],
        }
    )

df_results = pd.DataFrame(results)
df_results.sort_values("test_pr_auc", ascending=False)

2026/06/10 20:22:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 20:22:15 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/10 20:22:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 20:22:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.h

,run_id,run_name,hidden_layers,learning_rate,epochs,dropout,train_roc_auc,test_roc_auc,roc_auc_gap,train_f1_score,test_f1_score,f1_score_gap,test_accuracy,test_precision,test_recall
2,4c1904cc504244e4a42af1a8501c996c,mlp_32_16_lr_001_epochs_75,"(32, 16)",0.0010,75,0.0,0.910261,0.842507,0.067754,0.698524,0.590529,0.107995,0.791341,0.616279,0.566845
1,b200bb62eaca47b3b0b1630bdcd2c399,mlp_64_32_lr_001_dropout_01_epochs_100,"(64, 32)",0.0010,100,0.1,0.934596,0.834958,0.099638,0.738197,0.563135,0.175062,0.786373,0.615873,0.518717
3,a8ce7ada0c4f4da788b10bffdc02b7ed,mlp_96_48_lr_001_dropout_01_epochs_75,"(96, 48)",0.0010,75,0.1,0.944386,0.832708,0.111678,0.763548,0.565158,0.198391,0.775018,0.580282,0.550802
0,7d4422b039824f32b3c73e3a9938c276,mlp_64_32_lr_0005_epochs_100,"(64, 32)",0.0005,100,0.0,0.928328,0.828443,0.099885,0.705271,0.567208,0.138063,0.792051,0.633663,0.513369


## Experimento com validação cruzada

Nesta etapa, os modelos são comparados usando apenas `X_train` e `y_train`. O conjunto `X_test` fica reservado para a avaliação final do melhor modelo.

In [7]:
cv_experiments = [
    {
        "run_name": "cv_mlp_256_128_32_lr_001_dropout_01",
        "hidden_layers": (256, 128, 32),
        "learning_rate": 0.001,
        "epochs": 50,
        "dropout": 0.1,
        "batch_size": 64,
    },
    {
        "run_name": "cv_mlp_128_64_32_16_lr_001_dropout_01",
        "hidden_layers": (128, 64, 32, 16),
        "learning_rate": 0.001,
        "epochs": 75,
        "dropout": 0.1,
        "batch_size": 64,
    },
    {
        "run_name": "cv_mlp_1024_512_256_128_lr_001_dropout_01",
        "hidden_layers": (1024, 512, 256, 128),
        "learning_rate": 0.001,
        "epochs": 75,
        "dropout": 0.1,
        "batch_size": 64,
    },
    {
        "run_name": "cv_mlp_128_64_lr_001_dropout_02",
        "hidden_layers": (512, 256, 128, 64),
        "learning_rate": 0.001,
        "epochs": 100,
        "dropout": 0.1,
        "batch_size": 64,
    },
]

In [9]:
import importlib

import experiment_tracker
import pandas as pd

importlib.reload(experiment_tracker)
from experiment_tracker import log_metrics_experiment

cv_tracking_results = []

for config in cv_experiments:
    cv_results, cv_summary = cross_validate_model(
        X=X_train,
        y=y_train,
        n_splits=5,
        hidden_layers=config["hidden_layers"],
        dropout=config["dropout"],
        learning_rate=config["learning_rate"],
        epochs=config["epochs"],
        batch_size=config["batch_size"],
    )

    cv_pr_auc_mean = cv_summary.loc["mean", "pr_auc"]
    cv_pr_auc_std = cv_summary.loc["std", "pr_auc"]

    metrics = {
        "cv_pr_auc_mean": cv_pr_auc_mean,
        "cv_pr_auc_std": cv_pr_auc_std,
        "pr_auc": cv_pr_auc_mean,
    }

    run_id = log_metrics_experiment(
        metrics=metrics,
        params={
            "validation_strategy": "stratified_kfold",
            "n_splits": 5,
            "hidden_layers": str(config["hidden_layers"]),
            "learning_rate": config["learning_rate"],
            "epochs": config["epochs"],
            "dropout": config["dropout"],
            "batch_size": config["batch_size"],
        },
        experiment_name="churn_validacao_cruzada_pr_auc",
        run_name=config["run_name"],
    )

    cv_tracking_results.append(
        {
            "run_id": run_id,
            "run_name": config["run_name"],
            **config,
            **metrics,
        }
    )

df_cv_results = pd.DataFrame(cv_tracking_results)
df_cv_results.sort_values("cv_pr_auc_mean", ascending=False)

,run_id,run_name,hidden_layers,learning_rate,epochs,dropout,batch_size,cv_pr_auc_mean,cv_pr_auc_std,pr_auc
0,f21b1cb076734618ad71a49f2b9e85a4,cv_mlp_256_128_32_lr_001_dropout_01,"(256, 128, 32)",0.001,50,0.1,64,0.554998,0.015383,0.554998
1,88c9f90fb87a4f03850322fe19ea69fb,cv_mlp_128_64_32_16_lr_001_dropout_01,"(128, 64, 32, 16)",0.001,75,0.1,64,0.545603,0.008382,0.545603
2,55eb181c07da436eaa9ef01fea571139,cv_mlp_1024_512_256_128_lr_001_dropout_01,"(1024, 512, 256, 128)",0.001,75,0.1,64,0.531156,0.034720,0.531156
3,2b6301ae18df4493ba7cd388305354ff,cv_mlp_128_64_lr_001_dropout_02,"(512, 256, 128, 64)",0.001,100,0.1,64,0.522162,0.028905,0.522162


In [10]:
best_cv_config = df_cv_results.sort_values(
    "cv_pr_auc_mean", ascending=False
).iloc[0]
best_cv_config

run_id               f21b1cb076734618ad71a49f2b9e85a4
run_name          cv_mlp_256_128_32_lr_001_dropout_01
hidden_layers                          (256, 128, 32)
learning_rate                                   0.001
epochs                                             50
dropout                                           0.1
batch_size                                         64
cv_pr_auc_mean                               0.554998
cv_pr_auc_std                                0.015383
pr_auc                                       0.554998
Name: 0, dtype: object

## Salvar melhor modelo

Treina o melhor modelo encontrado na validação cruzada usando o conjunto de treino completo e salva o pacote em `models/modelo_v1.joblib`.

In [11]:
import ast
from pathlib import Path

import joblib
from data_ingestion import MODEL_FEATURES

hidden_layers = best_cv_config["hidden_layers"]
if isinstance(hidden_layers, str):
    hidden_layers = ast.literal_eval(hidden_layers)

best_model, best_scaler, best_losses = fit_model(
    X_train=X_train,
    y_train=y_train,
    hidden_layers=hidden_layers,
    learning_rate=float(best_cv_config["learning_rate"]),
    epochs=int(best_cv_config["epochs"]),
    dropout=float(best_cv_config["dropout"]),
    batch_size=int(best_cv_config["batch_size"]),
)

best_test_metrics = evaluate_model(
    model=best_model,
    scaler=best_scaler,
    X_test=X_test,
    y_test=y_test,
)

project_root = Path.cwd().resolve()
while (
    project_root.name != "tech-challenge-fase1"
    and project_root != project_root.parent
):
    project_root = project_root.parent

model_path = project_root / "models" / "modelo_v1.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)

model_package = {
    "model": best_model,
    "scaler": best_scaler,
    "features": MODEL_FEATURES,
    "threshold": 0.5,
    "metrics": best_test_metrics,
    "losses": best_losses,
    "config": {
        "hidden_layers": hidden_layers,
        "learning_rate": float(best_cv_config["learning_rate"]),
        "epochs": int(best_cv_config["epochs"]),
        "dropout": float(best_cv_config["dropout"]),
        "batch_size": int(best_cv_config["batch_size"]),
        "cv_pr_auc_mean": float(best_cv_config["cv_pr_auc_mean"]),
        "cv_pr_auc_std": float(best_cv_config["cv_pr_auc_std"]),
    },
}

joblib.dump(model_package, model_path)

{
    "model_path": str(model_path),
    "test_metrics": best_test_metrics,
}

{'model_path': 'C:\\Users\\Lucas\\Desktop\\pos_tech\\tech-challenge-fase1\\models\\modelo_v1.joblib',
 'test_metrics': {'pr_auc': 0.561106006026123}}

## Teste request pro modelo v1

In [12]:
import json
from pathlib import Path

import joblib
import pandas as pd
from model_trainer import predict, predict_proba

project_root = Path.cwd().resolve()
while (
    project_root.name != "tech-challenge-fase1"
    and project_root != project_root.parent
):
    project_root = project_root.parent

model_path = project_root / "models" / "modelo_v1.joblib"
json_path = project_root / "docs" / "examples" / "predict_input_model_v1.json"

model_package = joblib.load(model_path)
model = model_package["model"]
scaler = model_package["scaler"]
features = model_package["features"]
threshold = model_package.get("threshold", 0.5)

with open(json_path, encoding="utf-8") as file:
    payload = json.load(file)

records = (
    payload["data"]
    if isinstance(payload, dict) and "data" in payload
    else payload
)
X_request = pd.DataFrame(records)

missing_features = [
    feature for feature in features if feature not in X_request.columns
]
if missing_features:
    raise ValueError(f"Features ausentes no JSON: {missing_features}")

X_request = X_request[features]

probabilities = predict_proba(model=model, scaler=scaler, X=X_request)
predictions = predict(
    model=model, scaler=scaler, X=X_request, threshold=threshold
)

pd.DataFrame(
    {
        "probabilidade_churn": probabilities,
        "predicao_churn": predictions,
    }
)

,probabilidade_churn,predicao_churn
0,0.845403,1


In [13]:
from pathlib import Path

import joblib
import pandas as pd
from model_trainer import predict, predict_proba

payload = {
    "senior_citizen": 0,
    "partner": 1,
    "dependents": 0,
    "tenure_months": 12,
    "is_new_customer": 0,
    "phone_services": 1,
    "multiples_lines": 0,
    "internet_dsl": 0,
    "internet_fiber": 1,
    "online_security": 0,
    "online_backup": 1,
    "device_protection": 0,
    "tech_support": 0,
    "streaming_tv": 1,
    "streaming_movies": 1,
    "contract_month_to_month": 1,
    "contract_one_year": 0,
    "contract_two_year": 0,
    "paperless_billing": 1,
    "payment_method_mailed_check": 0,
    "payment_method_electronic_check": 1,
    "payment_method_bank_transfer_automatic": 0,
    "payment_method_credit_card_automatic": 0,
    "monthly_charges": 89.9,
    "total_charges": 1078.8,
    "avg_ticket": 89.9,
    "high_risk_profile": 1,
}

project_root = Path.cwd().resolve()
while (
    project_root.name != "tech-challenge-fase1"
    and project_root != project_root.parent
):
    project_root = project_root.parent

model_package = joblib.load(project_root / "models" / "modelo_v1.joblib")
model = model_package["model"]
scaler = model_package["scaler"]
features = model_package["features"]
threshold = model_package.get("threshold", 0.5)

X_request = pd.DataFrame([payload])

missing_features = [
    feature for feature in features if feature not in X_request.columns
]
if missing_features:
    raise ValueError(f"Features ausentes no JSON: {missing_features}")

X_request = X_request[features]

probabilities = predict_proba(model=model, scaler=scaler, X=X_request)
predictions = predict(
    model=model, scaler=scaler, X=X_request, threshold=threshold
)

pd.DataFrame(
    {
        "probabilidade_churn": probabilities,
        "predicao_churn": predictions,
    }
)

,probabilidade_churn,predicao_churn
0,0.845403,1
